### Capstone Assignment: Can objects in images be reliably determined using YOLO (You Only Look Once) methods?

**Context**

Imagine an autonomous vehicle driving through town. It needs to correctly identify other vehicles, traffic signs, traffic lights, people, etc., simply by looking at them. Varying weather, sunlight conditions, etc. can not be allowed to interfer. Can this be done within a reliable tollerance?

**Overview**

The goal of this project is to test the reliability of an ML model trained with ~70,000 images in hopes of it being able to distinguish between important objects and background noise.

**Data**

This dataset comes from the original Berkeley DeepDrive (BDD100K) dataset, converted into YOLO-compatible annotations. The images were captured for the purpose of training and testing artificial intelligence, so the images are all from ground level, in a variety of weather conditions including sunny, overcast and rainy, differing sunlight conditions including both daytime and nighttime, and from a variety of locations including New York, San Francisco, Berkley, and the Bay Area.

### Download the dataset

In [1]:
import kagglehub

path = kagglehub.dataset_download("a7madmostafa/bdd100k-yolo")

print("Path to dataset files:", path)

100%|██████████| 5.33G/5.33G [00:30<00:00, 188MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/a7madmostafa/bdd100k-yolo/versions/4


### Filter the dataset

The original dataset has 10 object types: person, rider, car, bus, truck, bike, motor, traffic signal, traffic sign, and train. For many of the object types there were few examples. For example, in the validate test set, there were only 649 instances of riders, 1597 instances of busses, 1007 bicycles, 452 motorcycles, and 15 trains.  Therefore, for this test we are only going to focus on some of the objects originally identified in the dataset, specifically cars, people, traffic signs, and traffic lights because these are the most common objects with at least 10,000 instances of each in the validation test set.

In [ ]:
import os
import shutil
from pathlib import Path

old_path = path
new_path = "/contents/temp"

# cleans the path we are going to be copying the subset of objects to
if not os.path.exists("/contents"):
    os.mkdir("/contents")
if os.path.exists(new_path):
    shutil.rmtree(new_path)
os.mkdir(new_path)

# creates the new yaml file for the training
def MakeYaml(yaml_path, old_names, new_ids):
    with open(yaml_path + "/data.yaml", "w") as f:
        f.write("path: " + yaml_path + "\n")
        f.write("train: " + yaml_path + "/train/images\n")
        f.write("val: " + yaml_path + "/val/images\n")
        f.write("test: " + yaml_path + "/test/images\n")
        print("nc:", len(new_ids), file=f)
        f.write("names:\n")
        for n in new_ids:
            f.write("  - " + old_names[n] + "\n")

# copies the image and labels for the training
def CopyData(old_path, new_path, old_names, new_ids):
    old_img_dir = old_path + "/images"
    old_label_dir = old_path + "/labels"
    old_ids = []
    for n in old_names:
        old_ids.append(-1)
    for n, id in enumerate(new_ids):
        old_ids[id] = n
    new_img_dir = new_path + "/images"
    new_label_dir = new_path + "/labels"
    os.mkdir(new_path)
    os.mkdir(new_img_dir)
    os.mkdir(new_label_dir)

    for item in Path(old_img_dir).iterdir():
        if item.is_file():
            sample_img_name = item.name
            old_img_path = os.path.join(old_img_dir, sample_img_name)
            old_label_path = os.path.join(old_label_dir, os.path.splitext(sample_img_name)[0] + ".txt")
            new_img_path = os.path.join(new_img_dir, sample_img_name)
            new_label_path = os.path.join(new_label_dir, os.path.splitext(sample_img_name)[0] + ".txt")

            with open(old_label_path, "r") as f:
                lines = f.readlines()

                will_copy = False
                for line in lines:
                    cls, x, y, bw, bh = map(float, line.strip().split())
                    cls = int(cls)
                    if old_ids[cls] != -1:
                        will_copy = True
                        break;
                if will_copy:
                    with open(new_label_path, "w") as f:
                        for line in lines:
                            cls, x, y, bw, bh = map(float, line.strip().split())
                            cls = int(cls)
                            if (old_ids[cls] != -1):
                                print(old_ids[cls], x, y, bw, bh, file=f)
                    shutil.copy(old_img_path, new_img_path)

# calls the other functions
def CopyAllData(old_path, new_path, old_names, new_ids):
    MakeYaml((new_path), old_names, new_ids)
    CopyData((old_path + "/train"), (new_path + "/train"), old_names, new_ids)
    CopyData((old_path + "/val"), (new_path + "/val"), old_names, new_ids)
    CopyData((old_path + "/test"), (new_path + "/test"), old_names, new_ids)

# copy the data, filtering only the desired object types
CLASS_NAMES = ["person", "rider", "car", "bus", "truck", "bike", "motor", "traffic light", "traffic sign", "train"]
CopyAllData(old_path, new_path, CLASS_NAMES, [0, 2, 7, 8])


### Train the YOLO model



In [ ]:
# clean the output directory
if os.path.exists("/content/runs"):
    shutil.rmtree("/content/runs")

# create and train the YOLO model
!pip install ultralytics
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
results = model.train(data=new_path + "/data.yaml", epochs=100, imgsz=640)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.8 MB/s eta 0:00:00a 0:00:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.25 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/contents/temp/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, f

### Validate the YOLO model



In [ ]:
#gather the latest scores on the validation test set
model.val()


Ultralytics 8.4.25 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2690.9±1017.2 MB/s, size: 54.7 KB)
val: Scanning /contents/temp/val/labels.cache... 10000 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 10000/10000 5.2Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 625/625 34.8it/s 18.0s<0.0s
                   all      10000     177611      0.715      0.562      0.624      0.313
                person       3220      13265      0.726      0.452      0.547      0.262
                   car       9879     102540      0.769      0.693      0.758      0.467
         traffic light       5653      26891      0.653      0.569      0.585      0.211
          traffic sign       8221      34915      0.714      0.533      0.606       0.31
Sp

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x79533ac1c950>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0

### Run inference on the YOLO model


In [ ]:
#scan the validation test set to find how many of each object type were found
results = model.predict(new_path + "/val/images/")


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/10000 /contents/temp/val/images/b1c66a42-6f7d68ca.jpg: 384x640 25 cars, 2 traffic lights, 3 traffic signs, 13.9ms
image 2/10000 /contents/temp/val/images/b1c81faa-3df17267.jpg: 384x640 2 cars, 1 traffic light, 2 traffic signs, 2.1ms
image 3/10000 /contents/temp/val/images/b1c81faa-c80764c5.jpg: 384x640 5 cars, 3 traffic signs, 2.2ms
image 4/10000 /contents/temp/val/images/b1c9c847-3bda4659.jpg: 384x640 19 cars, 1.9ms
image 5/10000 /contents/temp/

### Results of the YOLO model run(s)



In [ ]:
#display the results of the validation test set after each epoch
import pandas as pd

# Path to YOLO results
results_path = "/content/runs/detect/train/results.csv"

# Load results
df = pd.read_csv(results_path)

# Display first few rows to confirm
display(df)

,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2,lr/pg3,lr/pg4,lr/pg5,lr/pg6,lr/pg7
0,1,296.119,1.60503,1.28595,1.03183,0.57570,0.42214,0.44492,0.21122,1.48312,0.91129,0.98412,0.009998,0.003333,0.009998,0.003333,0.009998,0.003333,0.009998,0.003333
1,2,530.478,1.56297,0.98748,1.01346,0.60774,0.44493,0.47816,0.22655,1.45966,0.86561,0.98147,0.019800,0.006600,0.019800,0.006600,0.019800,0.006600,0.019800,0.006600
2,3,743.125,1.56863,0.96599,1.01736,0.61799,0.46066,0.49577,0.23662,1.44954,0.84266,0.98149,0.029404,0.009801,0.029404,0.009801,0.029404,0.009801,0.029404,0.009801
3,4,948.659,1.54817,0.93924,1.01109,0.64479,0.47868,0.52253,0.25073,1.41799,0.80620,0.96971,0.029109,0.009703,0.029109,0.009703,0.029109,0.009703,0.029109,0.009703
4,5,1154.040,1.51518,0.90019,0.99807,0.65142,0.49333,0.53798,0.25960,1.39947,0.78349,0.96285,0.028812,0.009604,0.028812,0.009604,0.028812,0.009604,0.028812,0.009604
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,19869.300,1.35617,0.71557,0.93721,0.71252,0.56016,0.62206,0.31103,1.31397,0.68821,0.92155,0.001785,0.000595,0.001785,0.000595,0.001785,0.000595,0.001785,0.000595
96,97,20072.200,1.35251,0.71236,0.93610,0.71286,0.56014,0.62212,0.31111,1.31396,0.68830,0.92145,0.001488,0.000496,0.001488,0.000496,0.001488,0.000496,0.001488,0.000496
97,98,20275.500,1.35047,0.70975,0.93504,0.71286,0.56021,0.62216,0.31116,1.31386,0.68817,0.92130,0.001191,0.000397,0.001191,0.000397,0.001191,0.000397,0.001191,0.000397
98,99,20477.600,1.34804,0.70752,0.93438,0.71330,0.56001,0.62225,0.31130,1.31374,0.68815,0.92115,0.000894,0.000298,0.000894,0.000298,0.000894,0.000298,0.000894,0.000298
